Figure out functions to process a user dataset and import it to the SQLite schema.

In [109]:
import pandas as pd
import sqlite3
from contextlib import contextmanager
from pathlib import Path
import uuid
import datetime
import csv

In [19]:
#Helpers

DB_FILE = Path('../sqlite_backend.db')


@contextmanager
def get_db_connection():
    conn = sqlite3.connect(DB_FILE)
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

Assume we receive this request:

In [96]:
request = {
    'parameters': {
        'uuid': None,#'0123456789ABCDEF0123456789ABCDEF', #Manually inserted as test user
        'datasetName':'MyTestDataset',
        'datasetType': 'FoldChange',#Curve/FoldChange
        'omics': 'Phosphorylation',#Phosphorylation/Other/Protein
        'hasFoldChangeColumn': 1, #1/0
        'foldChangeDataFoldChangeScale': 'raw', #raw/log/none
        'taxcode': 9606, #9606/10090
        
    },
    'file': {
        #TODO: See if this is possible or if you can only do a list
        'csvFile': '/home/jmueller/Downloads/examples/fold_change_data/volcano_ptm_data.csv'
        # 'tomlFile': 
    }
}
request

{'parameters': {'uuid': None,
  'datasetName': 'MyTestDataset',
  'datasetType': 'FoldChange',
  'omics': 'Phosphorylation',
  'hasFoldChangeColumn': 1,
  'foldChangeDataFoldChangeScale': 'raw',
  'taxcode': 9606},
 'file': {'csvFile': '/home/jmueller/Downloads/examples/fold_change_data/volcano_ptm_data.csv'}}

We start with obtaining a user id, optionally by creating it if it does not exist yet.

In [69]:
def getUserIdFromUUID(input_uuid):
    if not input_uuid:
        return None
    with get_db_connection() as conn:
        res = conn.execute('SELECT U.USER_ID FROM USER U WHERE U.SESSION_ID = ?', [input_uuid]).fetchall()
        if len(res) > 0:
            return res[0][0]
        else:
            return None

In [85]:
def getOrCreateUserId(input_uuid):
    user_id = getUserIdFromUUID(input_uuid)
    if not user_id:
        #Create new and insert
        new_uuid = str(uuid.uuid4()).replace('-', '').upper()
        with get_db_connection() as conn:
            conn.execute('INSERT INTO USER(SESSION_ID,LAST_ACCESSION_DATE) VALUES (?,?)', [new_uuid, datetime.datetime.now().isoformat()])
        user_id = getUserIdFromUUID(new_uuid)
    return user_id

In [90]:
user_id = getOrCreateUserId(request['parameters']['uuid'])
user_id

2

Now we create an entry into the user_dataset table:

In [102]:
def insertToUserDataset(dataset_name, user_id, dataset_type, omics, taxcode):
    with get_db_connection() as conn:
        conn.execute('INSERT INTO USER_DATASET(NAME, USER_ID, DATASET_TYPE, OMICS, TAXCODE) VALUES (?,?,?,?,?)',
                     [dataset_name, user_id, dataset_type, omics, taxcode])
        #Maybe necessary later: Return the id of the new dataset
        dataset_id = conn.execute('SELECT UD.DATASET_ID FROM USER_DATASET UD WHERE UD.USER_ID = ? AND UD.NAME = ?', 
                                  [user_id, dataset_name]
                                 ).fetchall()[0][0]
        return dataset_id

In [103]:
dataset_id = insertToUserDataset(
    request['parameters']['datasetName'],
    user_id,
    request['parameters']['datasetType'],
    request['parameters']['omics'],
    request['parameters']['taxcode'],
)
dataset_id

1

TODO: Figure out how to access the file,   
PrDB does it with         `internals.csvLines = request.entities[1].body.asString().split(/\r\n|\n/);`  
Enrichment Server does it with:
```
file = post_request.files['file']
input_filepath = output_dir / 'input.csv'
file.save(input_filepath)  
delimiter = get_delimiter(input_filepath) #So I wouldn't actually need the delimiter as a param, noice
input_df = pd.read_csv(input_filepath, sep=str(delimiter))

We assume we have that figured out and just parse the file into pandas

In [107]:
def get_delimiter(file_path, bytes=4096):
    sniffer = csv.Sniffer()
    data = open(file_path, "r").read(bytes)
    delimiter = sniffer.sniff(data).delimiter
    return delimiter

In [111]:
csv_delimiter = get_delimiter(request['file']['csvFile'])

In [112]:
input_csv_df = pd.read_csv(request['file']['csvFile'], sep=csv_delimiter)
input_csv_df

,Modified sequence,Protein IDs,Regulation,Experiment,Gene names,Log Fold Change,Adjusted pvalue
0,AAAAAAAGDS(ph)DSWDADAFSVEDPVRK,O75822,NaN,MyExperiment,EIF3J,-5.345297,0.458331
1,AAAAAAAGDSDS(ph)WDADAFSVEDPVRK,O75822,NaN,MyExperiment,EIF3J,-0.361372,0.967921
2,AAAAAAGPSPGSGPGDS(ph)PEGPEGEAPERR,Q9UID3,NaN,MyExperiment,VPS51,0.767573,0.906945
3,AAAAAALS(ph)GAGTPPAGGGAGGGGAGGGGS(ph)PPGGWAVAR,Q01167,NaN,MyExperiment,FOXK2,-0.768159,0.657085
4,AAAAAALSGAGT(ph)PPAGGGAGGGGAGGGGS(ph)PPGGWAVAR,Q01167,NaN,MyExperiment,FOXK2,-2.303435,0.610341
...,...,...,...,...,...,...,...
7867,YVSGS(ph)SPDLVTRK,Q15678,NaN,MyExperiment,PTPN14,0.105275,0.851965
7868,YVSGSS(ph)PDLVTRK,Q15678,NaN,MyExperiment,PTPN14,-0.338962,0.372046
7869,YVTKPNS(ph)DDEDDGDEK,P51784,NaN,MyExperiment,USP11,-0.443683,0.591849
7870,YWGPAS(ph)PTHK,Q9H6S3,NaN,MyExperiment,EPS8L2,-2.232358,0.562901


TODO: 
-  Fuzzy Match of Column Names, rename potentially
-  Check for Expected Columns Names, exit if one is missing 

Now we get a mapping of modification symbol to modification id

In [126]:
def get_modification_symbol_to_id():
    with get_db_connection() as conn:
        res = conn.execute('SELECT SYMBOL, MODIFICATION_ID FROM MODIFICATION').fetchall()
        return {symbol : modification_id for symbol, modification_id in res}

In [127]:
modification_symbol_to_id = get_modification_symbol_to_id()
modification_symbol_to_id

{'(ph)': 1}

# I. PTM
## A. Peptide
### 1. FoldChange

### 2. Curve

Parse the toml file from request['file']['toml']

## B. Site

# II. Protein